# Conferência do CO 1001

Este notebook faz 4 etapas:

1. Lê a matriz MSC corretamente (`skiprows=1`).
2. Filtra na matriz apenas `saldo final` do `CO 1001` (`TIPO4 = CO` e `IC4 = 1001`).
3. Constrói no arquivo do sistema as chaves equivalentes da MSC (`IC2 = funcao + sub_funcao` e `IC3 = ano_fonte + fonte`).
4. Aplica no extrato detalhado do sistema a mesma regra usada no relatório.
5. Compara os valores da matriz com o resultado filtrado do sistema.

Ajuste apenas os nomes dos arquivos e, se necessário, os aliases das colunas do extrato do sistema.

In [61]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.float_format = lambda x: f'{x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')


def normalizar_texto(texto):
    texto = '' if texto is None else str(texto).strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    for alvo in ['[', ']', '(', ')', '.', '-', '/', '\\']:
        texto = texto.replace(alvo, ' ')
    return ' '.join(texto.upper().split())


def localizar_colunas(df, aliases):
    mapa = {normalizar_texto(col): col for col in df.columns}
    encontradas = {}

    for destino, candidatos in aliases.items():
        for candidato in candidatos:
            chave = normalizar_texto(candidato)
            if chave in mapa:
                encontradas[destino] = mapa[chave]
                break

    faltantes = [col for col in aliases if col not in encontradas]
    if faltantes:
        raise KeyError(f'Colunas nao encontradas no extrato do sistema: {faltantes}')

    return df.rename(columns={origem: destino for destino, origem in encontradas.items()})


def converter_valor(serie):
    serie = serie.astype(str).str.strip()
    serie = serie.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

    tem_virgula = serie.str.contains(',', na=False)
    tem_ponto = serie.str.contains('.', regex=False, na=False)

    serie = np.where(
        tem_virgula & tem_ponto,
        serie.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        np.where(
            tem_virgula,
            serie.str.replace(',', '.', regex=False),
            serie,
        ),
    )

    return pd.to_numeric(pd.Series(serie), errors='coerce')


In [62]:
ano = '2026'
mes = '02'

arquivo_matriz = Path(f'msc_{mes}_{ano}.csv')
arquivo_sistema = Path('consulta_sistema_co1001.csv')

co_alvo = '1001'
tipo_co = 'CO'
tipo_valor_matriz = 'ending_balance'


In [63]:
# A matriz tem uma primeira linha com identificacao do arquivo, por isso usamos skiprows=1.
df_matriz = pd.read_csv(
    arquivo_matriz,
    sep=';',
    dtype=str,
    encoding='latin-1',
    skiprows=1,
)

df_matriz.columns = [col.strip() for col in df_matriz.columns]
for col in df_matriz.columns:
    df_matriz[col] = df_matriz[col].astype(str).str.strip()

df_matriz['VALOR_NUM'] = converter_valor(df_matriz['VALOR'])

matriz_co1001 = df_matriz.loc[
    (df_matriz['TIPO_VALOR'] == tipo_valor_matriz)
    & (df_matriz['TIPO4'] == tipo_co)
    & (df_matriz['IC4'] == co_alvo)
].copy()

# Chaves equivalentes para comparacao com o arquivo do sistema.
matriz_co1001['ic2_equivalente'] = matriz_co1001['IC2'].str.zfill(5)
matriz_co1001['ic3_equivalente'] = matriz_co1001['IC3'].str.zfill(4)
matriz_co1001['natureza_despesa_codigo'] = matriz_co1001['IC5'].str.zfill(8)
matriz_co1001['elemento_despesa_codigo'] = matriz_co1001['IC5'].str[-2:]

print('Linhas da matriz para CO 1001 (saldo final):', len(matriz_co1001))
print('Total da matriz para CO 1001 (saldo final):', matriz_co1001['VALOR_NUM'].sum())

matriz_co1001.head()


Linhas da matriz para CO 1001 (saldo final): 879
Total da matriz para CO 1001 (saldo final): 12839974512.91


,CONTA,IC1,TIPO1,IC2,TIPO2,IC3,TIPO3,IC4,TIPO4,IC5,...,IC6,TIPO6,VALOR,TIPO_VALOR,NATUREZA_VALOR,VALOR_NUM,ic2_equivalente,ic3_equivalente,natureza_despesa_codigo,elemento_despesa_codigo
25065,522110100,10111,PO,12243,FS,1500,FR,1001,CO,44905200,...,nan,nan,6700000,ending_balance,D,"6.700.000,00",12243,1500,44905200,00
25066,522110100,10111,PO,12364,FS,1500,FR,1001,CO,33903900,...,nan,nan,6179825,ending_balance,D,"6.179.825,00",12364,1500,33903900,00
25074,522110100,10111,PO,12368,FS,1500,FR,1001,CO,33903200,...,nan,nan,10000,ending_balance,D,"10.000,00",12368,1500,33903200,00
25403,522110100,10111,PO,12368,FS,1500,FR,1001,CO,33903900,...,nan,nan,392254764,ending_balance,D,"392.254.764,00",12368,1500,33903900,00
25407,522110100,10111,PO,12302,FS,1500,FR,1001,CO,44905100,...,nan,nan,1107418,ending_balance,D,"1.107.418,00",12302,1500,44905100,00


In [64]:
# Leia aqui a consulta detalhada extraida do sistema com as colunas da regra.
# Se o nome do arquivo for outro, altere apenas a variavel arquivo_sistema.
df_sistema = pd.read_csv(arquivo_sistema, sep=';', dtype=str, encoding='latin-1', quotechar='"')
df_sistema.columns = [col.strip() for col in df_sistema.columns]

aliases = {
    'fonte_rj': ['FONTE RJ', '[FONTE RJ].[CODIGO]', 'FONTE RJ CODIGO', 'FONTE_RJ', 'CODIGO FONTE RJ'],
    'fonte_stn': ['FONTE STN', 'FONTE_STN', 'FONTE NORMAL STN', 'FONTE NORMAL', 'FONTE'],
    'funcao_codigo': ['FUNCAO', '[FUNCAO].[CODIGO]', 'FUNCAO CODIGO', 'CODIGO FUNCAO'],
    'sub_funcao_codigo': ['SUB FUNCAO', 'SUB_FUNCAO', '[SUB FUNCAO].[CODIGO]', 'SUB FUNCAO CODIGO', 'CODIGO SUB FUNCAO'],
    'ano_fonte_codigo': ['ANO FONTE', 'ANO_FONTE', '[ANO FONTE].[CODIGO]', 'ANO FONTE CODIGO', 'CODIGO ANO FONTE'],
    'unidade_orcamentaria_codigo': ['UNIDADE_ORCAMENTARIA', 'UNIDADE ORCAMENTARIA', '[UNIDADE ORCAMENTARIA].[CODIGO]', 'UNIDADE ORCAMENTARIA CODIGO', 'CODIGO UNIDADE ORCAMENTARIA'],
    'unidade_gestora_saldo_codigo': ['UNIDADE_GESTORA_DO_SALDO', 'UNIDADE GESTORA DO SALDO', '[UNIDADE GESTORA DO SALDO].[CODIGO]', 'UNIDADE GESTORA DO SALDO CODIGO', 'CODIGO UNIDADE GESTORA DO SALDO'],
    'natureza_despesa_codigo': ['NATUREZA_DESPESA_8_DIGITOS', 'NATUREZA DA DESPESA 8 DIGITOS', '[NATUREZA DA DESPESA 8 DIGITOS].[CODIGO]', 'NATUREZA DA DESPESA CODIGO', 'CODIGO NATUREZA DA DESPESA 8 DIGITOS'],
    'acao_codigo': ['ACAO', '[ACAO].[CODIGO]', 'ACAO CODIGO', 'CODIGO ACAO'],
    'valor': ['VALOR', 'VALOR LIQUIDO', 'VALOR TOTAL'],
}

df_sistema = localizar_colunas(df_sistema, aliases)

for col in df_sistema.columns:
    df_sistema[col] = df_sistema[col].astype(str).str.strip()

df_sistema['valor_num'] = converter_valor(df_sistema['valor'])
df_sistema['funcao_codigo'] = df_sistema['funcao_codigo'].str.zfill(2)
df_sistema['sub_funcao_codigo'] = df_sistema['sub_funcao_codigo'].str.zfill(3)
df_sistema['ano_fonte_codigo'] = df_sistema['ano_fonte_codigo'].str.zfill(1)
df_sistema['fonte_rj'] = df_sistema['fonte_rj'].str.zfill(3)
df_sistema['fonte_stn'] = df_sistema['fonte_stn'].str.zfill(3)
df_sistema['ic2_equivalente'] = df_sistema['funcao_codigo'] + df_sistema['sub_funcao_codigo']
df_sistema['ic3_equivalente'] = df_sistema['ano_fonte_codigo'] + df_sistema['fonte_stn']
df_sistema['natureza_despesa_codigo'] = df_sistema['natureza_despesa_codigo'].str.zfill(8)
df_sistema['elemento_despesa_codigo'] = df_sistema['natureza_despesa_codigo'].str[2:4]

fontes_validas = ('100', '102', '107', '108', '122', '215')
uos_bloqueadas = ('1241', '2041', '2141', '40410')
nds_bloqueadas = (
    '319001', '319003', '319005', '31900703', '31901308', '31901312',
    '3370', '33900803', '339008', '33903992', '33904723', '339059',
    '33909302', '44909302'
)
acoes_bloqueadas = {'2253', '2701', '8302'}

mask_fonte = df_sistema['fonte_rj'].str.endswith(fontes_validas, na=False)
mask_funcao = df_sistema['funcao_codigo'].str[:2].eq('12')
mask_uo = df_sistema['unidade_orcamentaria_codigo'].str.startswith(uos_bloqueadas, na=False)
mask_ugs = df_sistema['unidade_gestora_saldo_codigo'].str.startswith('1234', na=False)
mask_nd = df_sistema['natureza_despesa_codigo'].str.startswith(nds_bloqueadas, na=False)
mask_elemento = df_sistema['elemento_despesa_codigo'].eq('92')
mask_acao = df_sistema['acao_codigo'].isin(acoes_bloqueadas)

mask_excecao = (
    mask_uo
    | mask_ugs
    | mask_nd
    | mask_elemento
    | mask_acao
)

sistema_co1001 = df_sistema.loc[mask_fonte & mask_funcao & ~mask_excecao].copy()

print('Total de linhas no CSV do sistema:', len(df_sistema))
print('Passam em FONTE_RJ:', int(mask_fonte.sum()))
print('Fontes STN distintas apos leitura:', df_sistema['fonte_stn'].nunique(dropna=True))
print('Passam em FUNCAO = 12:', int(mask_funcao.sum()))
print('Bloqueadas por UO:', int(mask_uo.sum()))
print('Bloqueadas por UGS saldo:', int(mask_ugs.sum()))
print('Bloqueadas por ND:', int(mask_nd.sum()))
print('Bloqueadas por Elemento 92:', int(mask_elemento.sum()))
print('Bloqueadas por Acao:', int(mask_acao.sum()))
print('Passam em FONTE_RJ + FUNCAO:', int((mask_fonte & mask_funcao).sum()))
print('Passam em tudo:', int((mask_fonte & mask_funcao & ~mask_excecao).sum()))

print('Linhas do sistema enquadradas na regra do CO 1001:', len(sistema_co1001))
print('Total do sistema para CO 1001:', sistema_co1001['valor_num'].sum())

print('\nAmostra de linhas que passaram em FONTE_RJ + FUNCAO mas foram excluidas pelas excecoes:')
df_sistema.loc[
    mask_fonte & mask_funcao & mask_excecao,
    [
        'CONTA', 'funcao_codigo', 'sub_funcao_codigo', 'ano_fonte_codigo', 'fonte_rj', 'fonte_stn',
        'unidade_orcamentaria_codigo', 'unidade_gestora_saldo_codigo',
        'natureza_despesa_codigo', 'elemento_despesa_codigo', 'acao_codigo', 'valor_num'
    ]
].head(20)


Total de linhas no CSV do sistema: 41186
Passam em FONTE_RJ: 22978
Fontes STN distintas apos leitura: 47
Passam em FUNCAO = 12: 4069
Bloqueadas por UO: 265
Bloqueadas por UGS saldo: 1865
Bloqueadas por ND: 1971
Bloqueadas por Elemento 92: 0
Bloqueadas por Acao: 18
Passam em FONTE_RJ + FUNCAO: 3021
Passam em tudo: 2856
Linhas do sistema enquadradas na regra do CO 1001: 2856
Total do sistema para CO 1001: 11957517897.13

Amostra de linhas que passaram em FONTE_RJ + FUNCAO mas foram excluidas pelas excecoes:


,CONTA,funcao_codigo,sub_funcao_codigo,ano_fonte_codigo,fonte_rj,fonte_stn,unidade_orcamentaria_codigo,unidade_gestora_saldo_codigo,natureza_despesa_codigo,elemento_despesa_codigo,acao_codigo,valor_num
1727,522110101,12,122,1,100,500,18010,180100,33900800,90,2660,"4.316.199,00"
1742,522110101,12,122,1,100,500,18020,210700,33900800,90,2660,"342.000,00"
1750,522110101,12,122,1,100,500,40430,404300,31900300,90,2660,"142.079,00"
1759,522110101,12,122,1,100,500,40430,404300,33900800,90,2660,"12.811.480,00"
1805,522110101,12,122,1,100,500,40440,404400,33900800,90,2019,"856.958,00"
1806,522110101,12,122,1,100,500,40440,404400,33900800,90,2022,"609.392,00"
1807,522110101,12,122,1,100,500,40440,404400,33900800,90,2660,"475.018,00"
1836,522110101,12,122,1,100,500,40450,404500,33900800,90,2660,"15.564.921,00"
1941,522110101,12,243,1,122,761,18020,210700,33903900,90,8302,"27.409.324,00"
1948,522110101,12,306,1,122,761,40440,404400,33903000,90,2253,"28.799.700,00"


In [65]:
# Comparacao principal:
# 1. CONTA + IC2 + IC3 + ND_8
# Comparacoes auxiliares:
# 2. Resumo por CONTA
# 3. IC2 + IC3
# 4. IC2 + IC3 + ND_6
# 5. IC2 + IC3 + ND_8
matriz_co1001['nd_6'] = matriz_co1001['natureza_despesa_codigo'].str[:6]
matriz_co1001['nd_8'] = matriz_co1001['natureza_despesa_codigo'].str[:8]
sistema_co1001['nd_6'] = sistema_co1001['natureza_despesa_codigo'].str[:6]
sistema_co1001['nd_8'] = sistema_co1001['natureza_despesa_codigo'].str[:8]

def comparar_nivel(df_matriz, df_sistema, chaves, rotulo):
    resumo_matriz = (
        df_matriz.groupby(chaves, dropna=False, as_index=False)['VALOR_NUM']
        .sum()
        .rename(columns={'VALOR_NUM': 'valor_matriz'})
    )

    resumo_sistema = (
        df_sistema.groupby(chaves, dropna=False, as_index=False)['valor_num']
        .sum()
        .rename(columns={'valor_num': 'valor_sistema'})
    )

    comparacao = resumo_matriz.merge(resumo_sistema, on=chaves, how='outer')
    comparacao['valor_matriz'] = comparacao['valor_matriz'].fillna(0)
    comparacao['valor_sistema'] = comparacao['valor_sistema'].fillna(0)
    comparacao['diferenca'] = comparacao['valor_matriz'] - comparacao['valor_sistema']
    comparacao['confere'] = np.isclose(comparacao['diferenca'], 0, atol=0.01)

    print(f'\n=== Comparacao: {rotulo} ===')
    print('Chaves       :', chaves)
    print('Total matriz :', resumo_matriz['valor_matriz'].sum())
    print('Total sistema:', resumo_sistema['valor_sistema'].sum())
    print('Dif. total   :', resumo_matriz['valor_matriz'].sum() - resumo_sistema['valor_sistema'].sum())
    print('Linhas ok    :', int(comparacao['confere'].sum()))
    print('Linhas diver.:', int((~comparacao['confere']).sum()))

    return comparacao.sort_values(['confere', 'diferenca'], ascending=[True, False])

print('\n=== Comparacao principal: CONTA + IC2 + IC3 + ND_8 ===')
comparacao_conta_ic2_ic3_nd8 = comparar_nivel(
    matriz_co1001,
    sistema_co1001,
    ['CONTA', 'ic2_equivalente', 'ic3_equivalente', 'nd_8'],
    'CONTA + IC2 + IC3 + ND_8'
)
comparacao_ic2_ic3_nd8 = comparacao_conta_ic2_ic3_nd8.copy()
comparacao_conta_ic2_ic3_nd8.head(50)

print('\n=== Resumo auxiliar por CONTA ===')
resumo_conta_matriz = (
    matriz_co1001.groupby(['CONTA'], dropna=False, as_index=False)['VALOR_NUM']
    .sum()
    .rename(columns={'VALOR_NUM': 'valor_matriz'})
)
resumo_conta_sistema = (
    sistema_co1001.groupby(['CONTA'], dropna=False, as_index=False)['valor_num']
    .sum()
    .rename(columns={'valor_num': 'valor_sistema'})
)
comparacao_por_conta = resumo_conta_matriz.merge(resumo_conta_sistema, on='CONTA', how='outer')
comparacao_por_conta['valor_matriz'] = comparacao_por_conta['valor_matriz'].fillna(0)
comparacao_por_conta['valor_sistema'] = comparacao_por_conta['valor_sistema'].fillna(0)
comparacao_por_conta['diferenca'] = comparacao_por_conta['valor_matriz'] - comparacao_por_conta['valor_sistema']
comparacao_por_conta['confere'] = np.isclose(comparacao_por_conta['diferenca'], 0, atol=0.01)
comparacao_por_conta = comparacao_por_conta.sort_values(['confere', 'diferenca'], ascending=[True, False])
comparacao_por_conta.head(50)

print('\n=== Comparacao auxiliar: IC2 + IC3 ===')
comparacao_ic2_ic3 = comparar_nivel(
    matriz_co1001,
    sistema_co1001,
    ['ic2_equivalente', 'ic3_equivalente'],
    'IC2 + IC3'
)

print('\n=== Comparacao auxiliar: IC2 + IC3 + ND_6 ===')
comparacao_ic2_ic3_nd6 = comparar_nivel(
    matriz_co1001,
    sistema_co1001,
    ['ic2_equivalente', 'ic3_equivalente', 'nd_6'],
    'IC2 + IC3 + ND_6'
)

print('\n=== Comparacao auxiliar: IC2 + IC3 + ND_8 ===')
comparacao_aux_ic2_ic3_nd8 = comparar_nivel(
    matriz_co1001,
    sistema_co1001,
    ['ic2_equivalente', 'ic3_equivalente', 'nd_8'],
    'IC2 + IC3 + ND_8'
)

contas_matriz = set(matriz_co1001['CONTA'].dropna().astype(str).unique())
contas_sistema = set(sistema_co1001['CONTA'].dropna().astype(str).unique())
contas_so_matriz = pd.DataFrame({'CONTA': sorted(contas_matriz - contas_sistema)})
contas_so_sistema = pd.DataFrame({'CONTA': sorted(contas_sistema - contas_matriz)})

print('\n=== Contas so na matriz ===')
print('Quantidade:', len(contas_so_matriz))
contas_so_matriz.head(50)

print('\n=== Contas so no sistema ===')
print('Quantidade:', len(contas_so_sistema))
contas_so_sistema.head(50)

print('\n=== Contas so no sistema com valor ===')
contas_so_sistema_com_valor = (
    sistema_co1001[sistema_co1001['CONTA'].astype(str).isin(contas_so_sistema['CONTA'])]
    .groupby(['CONTA'], dropna=False, as_index=False)['valor_num']
    .sum()
    .rename(columns={'valor_num': 'valor_sistema'})
    .sort_values('valor_sistema', ascending=False)
)
contas_so_sistema_com_valor.head(50)

print('\n=== Maiores diferencas por CONTA ===')
comparacao_por_conta.assign(abs_diferenca=lambda df: df['diferenca'].abs())\
    .sort_values('abs_diferenca', ascending=False)\
    .drop(columns=['abs_diferenca'])\
    .head(50)

print('\nTop 50 divergencias da comparacao principal (CONTA + IC2 + IC3 + ND_8):')
comparacao_ic2_ic3_nd8.head(50)



=== Comparacao principal: CONTA + IC2 + IC3 + ND_8 ===

=== Comparacao: CONTA + IC2 + IC3 + ND_8 ===
Chaves       : ['CONTA', 'ic2_equivalente', 'ic3_equivalente', 'nd_8']
Total matriz : 12839974512.91
Total sistema: 11957517897.13
Dif. total   : 882456615.7800007
Linhas ok    : 666
Linhas diver.: 1860

=== Resumo auxiliar por CONTA ===

=== Comparacao auxiliar: IC2 + IC3 ===

=== Comparacao: IC2 + IC3 ===
Chaves       : ['ic2_equivalente', 'ic3_equivalente']
Total matriz : 12839974512.91
Total sistema: 11957517897.130001
Dif. total   : 882456615.7799988
Linhas ok    : 21
Linhas diver.: 12

=== Comparacao auxiliar: IC2 + IC3 + ND_6 ===

=== Comparacao: IC2 + IC3 + ND_6 ===
Chaves       : ['ic2_equivalente', 'ic3_equivalente', 'nd_6']
Total matriz : 12839974512.91
Total sistema: 11957517897.130001
Dif. total   : 882456615.7799988
Linhas ok    : 163
Linhas diver.: 31

=== Comparacao auxiliar: IC2 + IC3 + ND_8 ===

=== Comparacao: IC2 + IC3 + ND_8 ===
Chaves       : ['ic2_equivalente', '

,CONTA,ic2_equivalente,ic3_equivalente,nd_8,valor_matriz,valor_sistema,diferenca,confere
2,522110100,12122,1500,31901100,"2.447.610.425,00","0,00","2.447.610.425,00",False
74,522110100,12362,1540,31901100,"2.209.971.376,00","0,00","2.209.971.376,00",False
66,522110100,12361,1540,31901100,"829.713.382,00","0,00","829.713.382,00",False
77,522110100,12362,1540,31911300,"425.113.459,00","0,00","425.113.459,00",False
9,522110100,12122,1500,31911300,"422.042.684,00","0,00","422.042.684,00",False
117,522110100,12368,1500,33903900,"392.254.764,00","0,00","392.254.764,00",False
121,522110100,12368,1761,33903900,"362.506.811,00","0,00","362.506.811,00",False
1845,622310000,12362,1540,31900000,"337.295.614,41","0,00","337.295.614,41",False
78,522110100,12362,1540,33904600,"299.863.706,00","0,00","299.863.706,00",False
21,522110100,12122,1500,33903900,"228.568.443,00","0,00","228.568.443,00",False
